# Entrainer un modèle de classification

Nous allons entraîner un modèle de classification d'images capable de trier les articles de presse en fonction de la présence d'une illustration. Nous utiliserons notamment un modèle de classification YOLO de taille minimale.

In [ ]:
%pip install scikit-learn
%pip install ultralytics
%pip install tqdm
%pip install pandas

In [ ]:
import requests
import random
import os
import time
import shutil

from sklearn.model_selection import train_test_split # Permet de spliter facilement un dataset 
from tqdm import tqdm

## 1. Récupération des annotations

Un classificateur catégorise les documents d'un ensemble de données. Parmi les classificateurs, on trouve le classificateur multi-étiquettes supervisé.

Nous avons répartit les documents en classes prédéfinies :

- Avec illustration
- Sans illustration

Ici, les annotations correspondent aux dossiers où sont stockées les images. Les classificateurs sont des modèles qui attribuent une étiquette à l'image entière, ce qui simplifie grandement la génération de données.

In [ ]:
!apt-get install -y git-lfs > /dev/null
!git lfs install

!rm -rf formationVision
!git clone https://huggingface.co/datasets/Anarchiviste/formationVision

%cd formationVision
!git lfs pull
%cd ..

In [ ]:
source = "./formationVision"

illustrations = os.listdir(os.path.join(source, 'illustrated/'))
textOnly = os.listdir(os.path.join(source, 'textonly/'))

trainIllustrations, valIllustrations = # Utilisons la fonction train_test_split() pour de déterminer la taille du jeu d'entraînement (80% des images totales) et la seed pour un aléatoire reproductible
trainTextOnly, valTextOnly = # Utilisons la fonction train_test_split() pour de déterminer la taille du jeu d'entraînement (80% des images totales) et la seed pour un aléatoire reproductible


In [ ]:
!rm -rf dataset/
!mkdir -p ./dataset/train/illustrated/
!mkdir -p ./dataset/train/textonly/
!mkdir -p ./dataset/val/illustrated/
!mkdir -p ./dataset/val/textonly/


In [ ]:
print(f"Nous aurons {len(trainIllustrations)} illustrations dans Train et {len(valIllustrations)} illustrations dans Val")
print(f"Nous aurons {len(trainTextOnly)} textes dans Train et {len(valTextOnly)} textes dans Val")

In [ ]:
copies = [
    (trainIllustrations, 'illustrated', './dataset/train/illustrated/'),
    (valIllustrations, 'illustrated', './dataset/val/illustrated/'),
    (trainTextOnly, 'textonly', './dataset/train/textonly/'),
    (valTextOnly, 'textonly', './dataset/val/textonly/')
]

# Mise en place d'un dataset de Train et de Val en attribuant les images à leurs dossiers
for files, folder_name, dest_dir in copies:
    # copions ici toutes les images à leurs dossiers parents. 

In [ ]:
from ultralytics import YOLO

# Nous importons un modèle Yolo 26 nano classificateur avec la bibliothèque d'Ultralytics
model = YOLO("yolo26n-cls.pt")

In [ ]:
dataset = "./dataset/"
model.train(
    data=dataset,
    flipud=0, fliplr=0,
    # Attribuons ici les hyperparamètres d'entraînement # attributions des hyperparamètres : 10 cycles, les images sont en tailles 640x640
    project="./",   # chemin relatif au dossier courant
    name="train",
)

# Faire tourner une inférence

L'inférence en intelligence artificielle est le processus par lequel un modèle entraîné applique les connaissances acquises pour analyser de nouvelles données et produire une prédiction, une classification ou une réponse.

In [ ]:
# Yolo rend 2 modèles (.pt), un nommé best.pt et un last.pt. Sur de longs cycles d'entraînement il faut éviter le surapprentissage, on charge donc le modèle avec les meilleurs valeures.
best = YOLO("./runs/classify/train/weights/best.pt")

In [ ]:
import os
import pandas as pd

# Dossier contenant les images de test
test_dir = "./formationVision/test/press/"

results_list = []
image_files = [f for f in os.listdir(test_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

# Exécuter l'inférence sur chaque image
for file_name in image_files:
    file_path = os.path.join(test_dir, file_name)
    results = best(file_path, verbose=False)

    for r in results:
        probs = r.probs
        top_class_idx = probs.top1
        top_class_name = r.names[top_class_idx]
        top_class_conf = float(probs.top1conf)

        results_list.append({
            "filename": file_name,
            "prediction": top_class_name,
            "confidence": top_class_conf
        })

# Convertir en DataFrame pandas et sauvegarder en CSV
df = pd.DataFrame(results_list)
df.sort_values(by='filename', ascending=False)
df.to_csv("predictions.csv", index=False)

print(f"Inférence terminée ! Les résultats ont été enregistrés dans 'predictions.csv'.")
print(df.head())